# SSAQS Global SHAP Stability Analysis

This notebook calculates global SHAP feature-ranking stability for the SSAQS next-observed questionnaire stress prediction task.

**Analysis design**
- 35 participants and 3,098 eligible observations.
- Five participant-held-out folds using the participant allocation associated with the primary prediction evaluation.
- Random Forest regression with 300 trees, minimum leaf size 5 and `max_features=0.8`.
- Numeric median imputation, missingness indicators and standardisation; categorical most-frequent imputation and one-hot encoding.
- SHAP values are calculated for all eligible held-out observations.

Before calculating SHAP values, the notebook compares predictions from the reconstructed pipeline with the archived out-of-fold predictions. The analysis proceeds only when the prediction agreement criterion is met. This comparison is a check on correspondence with the primary prediction analysis; it is not a test of external validity.

The SSAQS analysis is independent of the RELAX analyses. SHAP values describe model attributions, not causal effects.

## 1. Computational environment

Use a Python environment with NumPy, pandas, scikit-learn, SHAP, SciPy and Matplotlib installed. The notebook records the active package versions in its analysis metadata. For exact numerical comparison, use the versions recorded with the supplied results.

In [ ]:
# Do not upgrade core numerical/ML packages during a reproducibility run.
# Record the active Python and scientific-library versions.
print('Package versions are recorded in the analysis metadata.')


## 2. Imports and configuration

In [ ]:
import os, zipfile, json, hashlib, itertools, warnings, sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import shap

warnings.filterwarnings("ignore", category=FutureWarning)
print("Python:", sys.version)
print("NumPy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", __import__("sklearn").__version__, "| SHAP:", shap.__version__)

SEED = 20261009
N_SPLITS = 5
TOP_K = 5
RF_CONFIG = dict(n_estimators=300, min_samples_leaf=5, max_features=0.8,
                 random_state=SEED, n_jobs=-1)

WORK_DIR = Path("SSAQS_Global_SHAP_Analysis")
EXTRACT_DIR = WORK_DIR / "source_extracted"
OUT_DIR = WORK_DIR / "outputs"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

The version-matched timezone-safe reference modelling table and archived primary OOF predictions are included in `inputs/`. Use `SSAQS_modelling_table_timezone_safe_v2_REFERENCE.csv`: its SHA-256 must be cfeade054717031fe56874516f123a68634953ba1e92252f025617279d2e705b. Do not substitute `SSAQS_modelling_table_reconstructed.csv`; that reconstructed file has lost the categorical activity-mode values and cannot reproduce the original predictor matrix.

This notebook uses the prepared modelling table for the model-aligned global SHAP analysis. Rebuilding the table from the raw participant archive is optional when the prepared table and corresponding prediction file are available.

In [ ]:
input_dir = Path('inputs')
source_csv = input_dir / 'SSAQS_modelling_table_timezone_safe_v2_REFERENCE.csv'
original_oof_path = input_dir / 'next_stress_groupcv_oof_predictions.csv'
for required in [source_csv, original_oof_path]:
    if not required.exists():
        raise FileNotFoundError(f'Required input not found: {required.resolve()}')
print('Selected modelling table:', source_csv.resolve())
print('Selected primary OOF predictions:', original_oof_path.resolve())

## 4. Reconstruct the original next-response task and feature set

The target is the next observed questionnaire stress response. Current-row stress and anxiety outcomes are not included as predictors.

In [ ]:
raw = pd.read_csv(source_csv)
raw["questionnaire_time_utc"] = pd.to_datetime(raw["questionnaire_time_utc"], utc=True, errors="coerce")
raw = raw.sort_values(["participant_id", "questionnaire_time_utc"]).reset_index(drop=True)
grouped = raw.groupby("participant_id", sort=False)
raw["target_next_stress"] = grouped["self_reported_stress"].shift(-1)
raw["next_questionnaire_time_utc"] = grouped["questionnaire_time_utc"].shift(-1)

df = raw.loc[
    raw["target_next_stress"].notna() &
    raw["next_questionnaire_time_utc"].notna()
].copy().reset_index(drop=True)

lag_num = [
    "previous_self_reported_stress",
    "previous_self_reported_anxiety",
    "days_since_previous_response"
]
sensor_num = [
    "steps_records_24h", "steps_sum_24h", "activity_records_24h",
    "oxygen_records_24h", "oxygen_mean_24h_valid85plus",
    "oxygen_records_below85_24h", "oxygen_records_24h_total",
    "sleep_records_48h", "sleep_overall_score_last_48h",
    "sleep_deep_minutes_last_48h"
]
sensor_cat = ["activity_level_mode_24h"]
numeric_features = lag_num + sensor_num
# Preserve the original primary-pipeline feature specification, including the
# nominal activity-mode column. This reference table contains observed activity
# modes; most-frequent imputation and one-hot encoding are fitted within each fold.
categorical_features = sensor_cat.copy()
features = numeric_features + categorical_features

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in categorical_features:
    df[col] = df[col].astype("object")

X = df[features].copy()
y = pd.to_numeric(df["target_next_stress"], errors="coerce").to_numpy(float)
# Keep participant IDs in their original numeric dtype. The verified allocation
# below is keyed by these original IDs; do not convert them to strings.
groups = df["participant_id"].to_numpy()

print("Eligible rows:", len(df))
print("Participants:", len(np.unique(groups)))
print("Predictors:", len(features), "| numeric:", len(numeric_features), "| categorical:", len(categorical_features))
print("Categorical observed values:", {c: int(df[c].notna().sum()) for c in categorical_features})
assert len(df) == 3098, f"Expected 3,098 eligible rows, found {len(df)}"
assert len(np.unique(groups)) == 35, f"Expected 35 participants, found {len(np.unique(groups))}"
assert np.isfinite(y).all()


## 5. Preprocessing

The preprocessing follows the primary SSAQS prediction pipeline: numeric median imputation with missingness indicators, followed by `StandardScaler`; categorical predictors use most-frequent imputation and one-hot encoding. Participant identifiers retain their original numeric data type because the original participant-to-fold allocation is used in the analysis.

In [ ]:
def make_original_preprocessor():
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

    transformers = [
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scale', StandardScaler())
        ]), numeric_features)
    ]
    if categorical_features:
        transformers.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', ohe)
        ]), categorical_features))
    return ColumnTransformer(transformers, remainder='drop')

In [ ]:
# Comparison of reconstructed and primary out-of-fold predictions
# This uses the verified original participant-to-fold allocation recovered by
# The participant-to-fold allocation from the primary evaluation is specified explicitly
# to preserve the original evaluation design across software versions.
if not original_oof_path.exists():
    raise FileNotFoundError(f"Primary OOF reference not found: {original_oof_path.resolve()}")
original_oof = pd.read_csv(original_oof_path)
for time_col in ["questionnaire_time_utc", "next_questionnaire_time_utc"]:
    original_oof[time_col] = pd.to_datetime(original_oof[time_col], utc=True, errors="coerce")

PARTICIPANT_TO_FOLD = {
    2: 1, 5: 1, 11: 1, 15: 1, 23: 1, 30: 1, 33: 1,
    1: 2, 8: 2, 9: 2, 10: 2, 18: 2, 28: 2, 31: 2,
    4: 3, 6: 3, 12: 3, 20: 3, 24: 3, 25: 3, 27: 3,
    7: 4, 13: 4, 14: 4, 21: 4, 22: 4, 32: 4, 34: 4,
    3: 5, 16: 5, 17: 5, 19: 5, 26: 5, 29: 5, 35: 5
}
actual_participants = set(int(v) for v in np.unique(groups))
if set(PARTICIPANT_TO_FOLD) != actual_participants:
    raise AssertionError(
        "Participant IDs do not match the verified original fold map. "
        f"Missing={sorted(actual_participants-set(PARTICIPANT_TO_FOLD))}; "
        f"unexpected={sorted(set(PARTICIPANT_TO_FOLD)-actual_participants)}"
    )
row_folds = np.array([PARTICIPANT_TO_FOLD[int(pid)] for pid in groups], dtype=int)
fixed_splits = []
for fold_id in range(1, N_SPLITS + 1):
    test_idx = np.flatnonzero(row_folds == fold_id)
    train_idx = np.flatnonzero(row_folds != fold_id)
    fixed_splits.append((train_idx, test_idx))
expected_fold_sizes = [627, 634, 602, 627, 608]
actual_fold_sizes = [len(te) for _, te in fixed_splits]
if actual_fold_sizes != expected_fold_sizes:
    raise AssertionError(
        f"Verified fold sizes do not match: {actual_fold_sizes}; expected {expected_fold_sizes}"
    )

comparison_predictions = np.full(len(df), np.nan, dtype=float)
comparison_fold = row_folds.copy()
for fold_id, (tr, te) in enumerate(fixed_splits, start=1):
    counts = pd.Series(groups[tr]).value_counts()
    weights = np.array([1.0 / counts[g] for g in groups[tr]], dtype=float)
    pre = make_original_preprocessor()
    Xtr = pre.fit_transform(X.iloc[tr])
    Xte = pre.transform(X.iloc[te])
    rf = RandomForestRegressor(**RF_CONFIG)
    rf.fit(Xtr, y[tr], sample_weight=weights)
    comparison_predictions[te] = rf.predict(Xte)

# Match rows by participant and questionnaire timestamps, not by assumed row order.
key_cols = ["participant_id", "questionnaire_time_utc", "next_questionnaire_time_utc"]
current_keys = df[key_cols].copy()
current_keys["current_position"] = np.arange(len(df))
original_keys = original_oof[key_cols].copy()
original_keys["original_oof_position"] = np.arange(len(original_oof))
matched = current_keys.merge(original_keys, on=key_cols, how="inner", validate="one_to_one")
if len(matched) != len(df) or len(original_oof) != len(df):
    raise AssertionError(
        f"OOF row matching failed: current={len(df)}, original={len(original_oof)}, matched={len(matched)}"
    )

orig_pred_col = "pred__Lagged questionnaire + sensors__Random Forest"
if orig_pred_col not in original_oof.columns:
    raise KeyError(f"Expected archived prediction column missing: {orig_pred_col}")
current_order = matched["current_position"].to_numpy()
original_order = matched["original_oof_position"].to_numpy()
original_predictions = original_oof.iloc[original_order][orig_pred_col].to_numpy(float)
reproduced_predictions = comparison_predictions[current_order]
comparison = pd.DataFrame({
    "participant_id": df.iloc[current_order]["participant_id"].to_numpy(),
    "questionnaire_time_utc": df.iloc[current_order]["questionnaire_time_utc"].astype(str).to_numpy(),
    "original_prediction": original_predictions,
    "reproduced_prediction": reproduced_predictions,
    "absolute_difference": np.abs(original_predictions - reproduced_predictions)
})
max_diff = float(comparison["absolute_difference"].max())
mean_diff = float(comparison["absolute_difference"].mean())
corr = float(np.corrcoef(original_predictions, reproduced_predictions)[0, 1])
comparison.to_csv(OUT_DIR / "SSAQS_prediction_comparison.csv", index=False)

print("Primary evaluation OOF file:", original_oof_path)
print("Matched observations:", len(matched))
print("Frozen fold sizes:", actual_fold_sizes)
print("Maximum absolute prediction difference:", max_diff)
print("Mean absolute prediction difference:", mean_diff)
print("Prediction correlation:", corr)

# Fail closed. No SHAP analysis may proceed unless the original pipeline is reproduced.
if not (max_diff <= 1e-8 and corr >= 0.999999999):
    raise RuntimeError(
        "prediction agreement check FAILED. SHAP calculations are stopped. "
        "Do not interpret SHAP stability results from this run."
    )

print("Prediction correspondence confirmed. Proceeding to SHAP calculation.")


### Feature specification and participant-to-fold allocation

The original primary prediction script includes `activity_level_mode_24h` as a categorical predictor and applies most-frequent imputation followed by one-hot encoding. The selected timezone-safe reference table contains 2,271 observed values in this column; the categorical predictor is retained and encoded within each training fold. The notebook uses the participant-to-fold allocation from the primary prediction evaluation. Prediction correspondence is assessed before SHAP extraction.


## 6. Fit five participant-held-out models and calculate held-out SHAP values

The participant-to-fold allocation from the primary evaluation, model configuration, participant-balanced training weights and training-fold preprocessing are used to calculate held-out SHAP values for all 3,098 eligible observations.


In [ ]:
fold_importance_rows = []
heldout_shap_rows = []
fold_summary_rows = []
prediction_rows = []

# `fixed_splits` is created only after the prediction agreement check passes.

for fold, (train_idx, test_idx) in enumerate(fixed_splits, start=1):
    print(f"Fold {fold}/{N_SPLITS}: fitting model and explaining {len(test_idx)} held-out rows")

    train_counts = pd.Series(groups[train_idx]).value_counts()
    weights = np.array([1.0 / train_counts[g] for g in groups[train_idx]], dtype=float)

    preprocessor = make_original_preprocessor()
    X_train = preprocessor.fit_transform(X.iloc[train_idx])
    X_test = preprocessor.transform(X.iloc[test_idx])
    feature_names = preprocessor.get_feature_names_out().astype(str)

    model = RandomForestRegressor(**RF_CONFIG)
    model.fit(X_train, y[train_idx], sample_weight=weights)

    predictions = model.predict(X_test)
    shap_values = np.asarray(
        shap.TreeExplainer(model).shap_values(X_test, check_additivity=False)
    )
    if shap_values.ndim == 3:
        shap_values = shap_values[..., 0]
    if shap_values.ndim != 2:
        raise ValueError(f"Unexpected SHAP shape in fold {fold}: {shap_values.shape}")

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    ranks = pd.Series(mean_abs_shap, index=feature_names).rank(
        ascending=False, method="average"
    )

    for feature, value, rank in zip(feature_names, mean_abs_shap, ranks):
        fold_importance_rows.append({
            "fold": fold, "feature": feature,
            "mean_abs_shap": float(value), "rank": float(rank),
            "n_test_rows_explained": int(len(test_idx)),
            "n_test_participants": int(len(np.unique(groups[test_idx])))
        })

    for j, row_idx in enumerate(test_idx):
        rec = {
            "fold": fold, "row_index": int(row_idx),
            "participant_id": groups[row_idx],
            "target_next_stress": float(y[row_idx]),
            "predicted_stress": float(predictions[j])
        }
        rec.update({
            f"shap__{feature}": float(value)
            for feature, value in zip(feature_names, shap_values[j])
        })
        heldout_shap_rows.append(rec)
        prediction_rows.append({
            "fold": fold, "row_index": int(row_idx),
            "participant_id": groups[row_idx],
            "target_next_stress": float(y[row_idx]),
            "predicted_stress": float(predictions[j])
        })

    fold_summary_rows.append({
        "fold": fold,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "train_participants": int(len(np.unique(groups[train_idx]))),
        "test_participants": int(len(np.unique(groups[test_idx]))),
        "n_estimators": RF_CONFIG["n_estimators"],
        "min_samples_leaf": RF_CONFIG["min_samples_leaf"],
        "max_features": RF_CONFIG["max_features"],
        "random_state": RF_CONFIG["random_state"],
        "numeric_scaler": "StandardScaler"
    })

print("Completed all five SSAQS folds.")

## 7. Pairwise global stability metrics

For each fold, features are ranked by mean absolute SHAP. Pairwise fold agreement is calculated using Spearman correlation, Kendall correlation and top-five Jaccard similarity.

In [ ]:
fi = pd.DataFrame(fold_importance_rows)
fi.to_csv(OUT_DIR / "SSAQS_global_SHAP_importance_by_fold.csv", index=False)

wide = fi.pivot(index="feature", columns="fold", values="mean_abs_shap")
rank_wide = wide.rank(axis=0, ascending=False, method="average")

pair_rows = []
for fold_a, fold_b in itertools.combinations(wide.columns, 2):
    ra, rb = rank_wide[fold_a], rank_wide[fold_b]
    spearman = spearmanr(ra, rb).statistic
    kendall = kendalltau(ra, rb).statistic
    top_a = set(wide[fold_a].nlargest(min(TOP_K, len(wide))).index)
    top_b = set(wide[fold_b].nlargest(min(TOP_K, len(wide))).index)
    union = top_a | top_b
    jaccard = len(top_a & top_b) / len(union) if union else np.nan
    pair_rows.append({
        "dataset": "SSAQS", "fold_a": int(fold_a), "fold_b": int(fold_b),
        "n_features_compared": int(len(wide)),
        "spearman_rank_correlation": float(spearman),
        "kendall_rank_correlation": float(kendall),
        f"top_{TOP_K}_jaccard": float(jaccard)
    })

pairs = pd.DataFrame(pair_rows)
pairs.to_csv(OUT_DIR / "SSAQS_global_SHAP_pairwise_stability.csv", index=False)

summary = (fi.groupby("feature")
    .agg(mean_abs_shap=("mean_abs_shap", "mean"),
         sd_abs_shap=("mean_abs_shap", "std"),
         mean_rank=("rank", "mean"),
         sd_rank=("rank", "std"),
         folds_top5=("rank", lambda x: int((x <= TOP_K).sum())),
         folds_present=("fold", "nunique"))
    .reset_index()
    .sort_values("mean_abs_shap", ascending=False))
summary.to_csv(OUT_DIR / "SSAQS_global_SHAP_summary.csv", index=False)

pair_summary = pd.DataFrame([{
    "dataset": "SSAQS",
    "mean_spearman": pairs["spearman_rank_correlation"].mean(),
    "median_spearman": pairs["spearman_rank_correlation"].median(),
    "mean_kendall": pairs["kendall_rank_correlation"].mean(),
    f"mean_top_{TOP_K}_jaccard": pairs[f"top_{TOP_K}_jaccard"].mean(),
    "n_fold_pairs": len(pairs)
}])
pair_summary.to_csv(OUT_DIR / "SSAQS_global_SHAP_stability_summary.csv", index=False)

heldout = pd.DataFrame(heldout_shap_rows)
heldout.to_csv(OUT_DIR / "SSAQS_global_SHAP_heldout_values.csv", index=False)
pd.DataFrame(fold_summary_rows).to_csv(OUT_DIR / "SSAQS_global_SHAP_fold_summary.csv", index=False)
pd.DataFrame(prediction_rows).to_csv(OUT_DIR / "SSAQS_global_SHAP_oof_predictions.csv", index=False)

display(pair_summary)
display(summary.head(15))

## 8. Analysis checks and figures

The following checks summarise observation coverage, fold representation, model configuration, participant separation and finite SHAP values.

In [ ]:
checks = []
def check(name, passed, detail):
    checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})

check("All eligible rows have held-out SHAP values",
      len(heldout) == len(df), f"{len(heldout)} SHAP rows; {len(df)} eligible rows")
check("All five folds represented",
      sorted(pd.DataFrame(fold_summary_rows)["fold"].tolist()) == [1,2,3,4,5],
      "Expected folds 1–5")
check("All models use 300 trees",
      all(x["n_estimators"] == 300 for x in fold_summary_rows), "n_estimators=300")
check("All models use StandardScaler",
      all(x["numeric_scaler"] == "StandardScaler" for x in fold_summary_rows),
      "Median imputation + missing indicators + StandardScaler")
check("No participant overlap",
      all(
          not (set(groups[tr]) & set(groups[te]))
          for tr, te in fixed_splits
      ), "GroupKFold participant separation")
check("All SHAP values finite",
      np.isfinite(heldout.filter(regex=r"^shap__").to_numpy()).all(),
      "No NaN or infinite SHAP values")
check("Ten pairwise fold comparisons",
      len(pairs) == 10, f"Observed {len(pairs)} comparisons")

checks_df = pd.DataFrame(checks)
assert checks_df["passed"].all(), "The analysis consistency conditions were not satisfied."

top = summary.head(15).sort_values("mean_abs_shap")
plt.figure(figsize=(9, 6))
plt.barh(top["feature"], top["mean_abs_shap"], xerr=top["sd_abs_shap"])
plt.xlabel("Mean absolute SHAP across held-out folds")
plt.ylabel("Feature")
plt.title("SSAQS global SHAP importance — original preprocessing")
plt.tight_layout()
plt.savefig(OUT_DIR / "SSAQS_global_SHAP_importance.png", dpi=240)
plt.show()

plt.figure(figsize=(8, 5))
plt.boxplot([
    pairs["spearman_rank_correlation"],
    pairs["kendall_rank_correlation"],
    pairs[f"top_{TOP_K}_jaccard"]
], tick_labels=["Spearman", "Kendall", "Top-5 Jaccard"])
plt.ylim(-1.05, 1.05)
plt.ylabel("Pairwise fold agreement")
plt.title("SSAQS global explanation stability — original preprocessing")
plt.tight_layout()
plt.savefig(OUT_DIR / "SSAQS_global_SHAP_stability.png", dpi=240)
plt.show()

## 9. Save analysis outputs and metadata

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

metadata = {
    "analysis": "SSAQS global SHAP stability",
    "source_file": source_csv.name,
    "source_sha256": sha256(source_csv),
    "rf_config": RF_CONFIG,
    "prediction_agreement": {"maximum_absolute_difference": max_diff, "mean_absolute_difference": mean_diff, "correlation": corr, "n_matched_observations": int(len(comparison))},
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": __import__("sklearn").__version__,
    "shap_version": shap.__version__,
    "n_splits": N_SPLITS,
    "fold_assignment": "participant-to-fold allocation from the primary prediction evaluation",
    "fold_sizes": actual_fold_sizes,
    "seed": SEED,
    "preprocessing": "numeric median imputation + missing indicators + StandardScaler; categorical most-frequent imputation + one-hot encoding",
    "target": "next observed questionnaire self-reported stress",
    "predictors": features,
    "note": "Use the version-matched timezone-safe reference table (SHA-256 cfeade054717031fe56874516f123a68634953ba1e92252f025617279d2e705b); do not substitute the reconstructed table. SSAQS is analysed independently; no cross-dataset transfer or external validation was performed."
}
(OUT_DIR / "SSAQS_analysis_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

report = [
    "SSAQS GLOBAL SHAP STABILITY ANALYSIS",
    "=" * 68,
    f"Rows: {len(df)}; participants: {len(np.unique(groups))}; predictors: {len(features)}",
    f"Random Forest: {RF_CONFIG}",
    "Preprocessing: median imputation + missingness indicators + StandardScaler; categorical imputation + one-hot encoding.",
    "",
    "GLOBAL STABILITY SUMMARY",
    pair_summary.to_string(index=False),
    "",
    "Interpretation: within-dataset global explanation stability only; no cross-dataset transfer or external validation."
]
(OUT_DIR / "SSAQS_analysis_report.txt").write_text("\n".join(report), encoding="utf-8")

zip_path = WORK_DIR / "SSAQS_Global_SHAP_Results.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUT_DIR))
print("Created:", zip_path)

In [ ]:
# Save the analysis outputs after the analysis has completed.
zip_path = WORK_DIR / "SSAQS_Global_SHAP_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(WORK_DIR))
print("Analysis outputs saved to:", zip_path.resolve())
